In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [6]:
import math, os
import numpy as np
import pandas as pd
from typing import List

import torch
import torch.nn.functional as F
from datasets import Dataset, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import ModelConfig
from tqdm.auto import tqdm

# ═══════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════
HUB_DATASET    = "august66/hh_helpfulness_qwen2.5_1.5b_generation"
REF_MODEL      = "august66/qwen2.5-1.5b-base-hh-helpful-sft"
DPO_MODEL      = "august66/hh_qwen_1.5b_sft_dpo_model"
TARGET_MODEL   = "august66/hh_qwen1.5_drpo_fixed_beta"

DATA_CACHE_DIR  = "/root/autodl-tmp/data_cache"
MODEL_CACHE_DIR = "/root/autodl-tmp/model_cache"

DPO_BETA    = 0.1
SCALE       = 1.0
CLIPBOUND   = 10.0
MAX_LENGTH  = 2048
BATCH_SIZE  = 4

TARGET_SHORT = TARGET_MODEL.split("/")[-1]
CSV_PATH     = f"{TARGET_SHORT}_is_detail.csv"

print(f"Dataset:      {HUB_DATASET}")
print(f"Ref model:    {REF_MODEL}")
print(f"DPO model:    {DPO_MODEL}")
print(f"Target model: {TARGET_MODEL}")
print(f"CSV output:   {CSV_PATH}")

Dataset:      august66/hh_helpfulness_qwen2.5_1.5b_generation
Ref model:    august66/qwen2.5-1.5b-base-hh-helpful-sft
DPO model:    august66/hh_qwen_1.5b_sft_dpo_model
Target model: august66/hh_qwen1.5_drpo_fixed_beta
CSV output:   hh_qwen1.5_drpo_fixed_beta_is_detail.csv


In [7]:
# ═══════════════════════════════════════════
# Load dataset
# ═══════════════════════════════════════════
ds = load_dataset(HUB_DATASET, split="train", cache_dir=DATA_CACHE_DIR)
print(f"Loaded {len(ds)} rows | Columns: {ds.column_names}")
print(f"Features: {ds.features}")
print(f"\nRow 0 sample:")
for k, v in ds[0].items():
    print(f"  {k}: {type(v).__name__} = {str(v)[:120]}")

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/27.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/46070 [00:00<?, ? examples/s]

Loaded 46070 rows | Columns: ['prompt', 'response_1', 'response_2', 'reward_1', 'reward_2', 'rank']
Features: {'prompt': List({'content': Value('string'), 'role': Value('string')}), 'response_1': List({'content': Value('string'), 'role': Value('string')}), 'response_2': List({'content': Value('string'), 'role': Value('string')}), 'reward_1': Value('float64'), 'reward_2': Value('float64'), 'rank': Value('int64')}

Row 0 sample:
  prompt: list = [{'content': 'Hi, I want to learn to play horseshoes. Can you teach me?', 'role': 'user'}, {'content': 'I can, but maybe
  response_1: list = [{'content': 'The traditional horseshoes are usually flat on the bottom and have a long spike on the edge.  The players 
  response_2: list = [{'content': 'For 6 or 8 horseshoes, you need 2 sets of 4 horseshoes (because there are 4 horseshoes per team), 2 horses
  reward_1: float = 21.75
  reward_2: float = 39.25
  rank: int = 0


In [8]:
# ═══════════════════════════════════════════
# Load models + define helpers
# ═══════════════════════════════════════════

def load_model(model_path):
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16, trust_remote_code=True, cache_dir=MODEL_CACHE_DIR,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        model_path, padding_side="left", truncation_side="left",
        use_fast=True, trust_remote_code=True, cache_dir=MODEL_CACHE_DIR,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    return model.cuda().eval(), tokenizer

ref_model, tokenizer = load_model(REF_MODEL)
dpo_model, _ = load_model(DPO_MODEL)
target_model, _ = load_model(TARGET_MODEL)
print("All models loaded.")


# ── Core functions ──

@torch.no_grad()
def get_sequence_logps(model, prompt_texts, full_texts, desc=""):
    all_logps = []
    for i in tqdm(range(0, len(full_texts), BATCH_SIZE), desc=desc):
        fb = full_texts[i:i+BATCH_SIZE]
        pb = prompt_texts[i:i+BATCH_SIZE]
        inputs = tokenizer(fb, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(model.device)
        plens = [len(tokenizer(p, add_special_tokens=True).input_ids) for p in pb]
        logits = model(**inputs).logits
        log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
        token_lps = log_probs.gather(2, inputs["input_ids"][:, 1:].unsqueeze(-1)).squeeze(-1)
        mask = inputs["attention_mask"][:, 1:].clone()
        for b, pl in enumerate(plens):
            mask[b, :max(0, pl-1)] = 0
        all_logps.extend((token_lps * mask).sum(-1).float().cpu().tolist())
    return all_logps


@torch.no_grad()
def get_sequence_kl(model_p, model_q, prompt_texts, full_texts, desc=""):
    all_kls = []
    for i in tqdm(range(0, len(full_texts), BATCH_SIZE), desc=desc):
        fb = full_texts[i:i+BATCH_SIZE]
        pb = prompt_texts[i:i+BATCH_SIZE]
        inputs = tokenizer(fb, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(model_p.device)
        plens = [len(tokenizer(p, add_special_tokens=True).input_ids) for p in pb]
        log_p = F.log_softmax(model_p(**inputs).logits[:, :-1, :], dim=-1)
        log_q = F.log_softmax(model_q(**inputs).logits[:, :-1, :], dim=-1)
        kl = F.kl_div(log_q, log_p, log_target=True, reduction="none").sum(-1)
        mask = inputs["attention_mask"][:, 1:].clone()
        for b, pl in enumerate(plens):
            mask[b, :max(0, pl-1)] = 0
        all_kls.extend((kl * mask).sum(-1).float().cpu().tolist())
    return all_kls


def clip_ratio(vals, lo=1.0/CLIPBOUND, hi=CLIPBOUND):
    return [max(lo, min(hi, v)) for v in vals]

def ratio_normal(reward, mu_ref, mu_theta):
    return [np.exp((-1/(2*SCALE**2)) * (-(r-mr)**2 + (r-mt)**2))
            for r, mr, mt in zip(reward, mu_ref, mu_theta)]

def ratio_laplace(reward, mu_ref, mu_theta):
    return [np.exp((-1/(2*SCALE)) * (-abs(r-mt) + abs(r-mr)))
            for r, mr, mt in zip(reward, mu_ref, mu_theta)]

print("Helper functions defined.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

All models loaded.
Helper functions defined.


In [9]:
# ═══════════════════════════════════════════
# Prepare texts
# ═══════════════════════════════════════════

def get_response_text(response_col):
    if isinstance(response_col, list):
        for msg in response_col:
            if isinstance(msg, dict) and msg.get("role") == "assistant":
                return msg.get("content", "")
        return " ".join(msg.get("content", "") for msg in response_col if isinstance(msg, dict))
    return str(response_col)

def format_prompt(prompt_msgs):
    return tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)

def format_full(prompt_msgs, response_text):
    full = prompt_msgs + [{"role": "assistant", "content": response_text}]
    return tokenizer.apply_chat_template(full, tokenize=False, add_generation_prompt=False)

resp_1_texts = [get_response_text(r) for r in ds["response_1"]]
resp_2_texts = [get_response_text(r) for r in ds["response_2"]]
prompt_texts = [format_prompt(p) for p in tqdm(ds["prompt"], desc="Prompts")]
full_texts_1 = [format_full(p, r) for p, r in tqdm(zip(ds["prompt"], resp_1_texts), total=len(ds), desc="Full r1")]
full_texts_2 = [format_full(p, r) for p, r in tqdm(zip(ds["prompt"], resp_2_texts), total=len(ds), desc="Full r2")]

print(f"Prepared {len(full_texts_1)} pairs")

Prompts:   0%|          | 0/46070 [00:00<?, ?it/s]

Full r1:   0%|          | 0/46070 [00:00<?, ?it/s]

Full r2:   0%|          | 0/46070 [00:00<?, ?it/s]

Prepared 46070 pairs


In [ ]:
# ═══════════════════════════════════════════
# Compute log-probs (3 models × 2 responses) + KL terms
# ═══════════════════════════════════════════

# Log-probs
r1_ref  = get_sequence_logps(ref_model,    prompt_texts, full_texts_1, desc="Ref r1")
r1_dpo  = get_sequence_logps(dpo_model,    prompt_texts, full_texts_1, desc="DPO r1")
r1_tgt  = get_sequence_logps(target_model, prompt_texts, full_texts_1, desc="Tgt r1")
r2_ref  = get_sequence_logps(ref_model,    prompt_texts, full_texts_2, desc="Ref r2")
r2_dpo  = get_sequence_logps(dpo_model,    prompt_texts, full_texts_2, desc="DPO r2")
r2_tgt  = get_sequence_logps(target_model, prompt_texts, full_texts_2, desc="Tgt r2")

# KL terms
r1_kl_ref_dpo   = get_sequence_kl(ref_model,    dpo_model, prompt_texts, full_texts_1, desc="KL(ref||dpo) r1")
r1_kl_tgt_ref   = get_sequence_kl(target_model, ref_model, prompt_texts, full_texts_1, desc="KL(θ||ref) r1")
r1_kl_tgt_dpo   = get_sequence_kl(target_model, dpo_model, prompt_texts, full_texts_1, desc="KL(θ||dpo) r1")
r2_kl_ref_dpo   = get_sequence_kl(ref_model,    dpo_model, prompt_texts, full_texts_2, desc="KL(ref||dpo) r2")
r2_kl_tgt_ref   = get_sequence_kl(target_model, ref_model, prompt_texts, full_texts_2, desc="KL(θ||ref) r2")
r2_kl_tgt_dpo   = get_sequence_kl(target_model, dpo_model, prompt_texts, full_texts_2, desc="KL(θ||dpo) r2")

# Derived quantities
r1_implicit_r = [DPO_BETA * (d - r) for d, r in zip(r1_dpo, r1_ref)]
r2_implicit_r = [DPO_BETA * (d - r) for d, r in zip(r2_dpo, r2_ref)]
r1_mu_ref     = [-DPO_BETA * k for k in r1_kl_ref_dpo]
r2_mu_ref     = [-DPO_BETA * k for k in r2_kl_ref_dpo]
r1_mu_theta   = [DPO_BETA * (kr - kd) for kr, kd in zip(r1_kl_tgt_ref, r1_kl_tgt_dpo)]
r2_mu_theta   = [DPO_BETA * (kr - kd) for kr, kd in zip(r2_kl_tgt_ref, r2_kl_tgt_dpo)]

print(f"Implicit reward r1: mean={np.mean(r1_implicit_r):.4f}, r2: mean={np.mean(r2_implicit_r):.4f}")
print(f"mu_ref r1: {np.mean(r1_mu_ref):.4f}, mu_theta r1: {np.mean(r1_mu_theta):.4f}")
print(f"mu_ref r2: {np.mean(r2_mu_ref):.4f}, mu_theta r2: {np.mean(r2_mu_theta):.4f}")

In [ ]:
# ═══════════════════════════════════════════
# Compute IS ratios + save all intermediates to CSV
# ═══════════════════════════════════════════

# Old ratio: π_θ/π_ref
r1_old = clip_ratio([np.exp(t - r) for t, r in zip(r1_tgt, r1_ref)])
r2_old = clip_ratio([np.exp(t - r) for t, r in zip(r2_tgt, r2_ref)])
# Normal ratio
r1_norm = clip_ratio(ratio_normal(r1_implicit_r, r1_mu_ref, r1_mu_theta))
r2_norm = clip_ratio(ratio_normal(r2_implicit_r, r2_mu_ref, r2_mu_theta))
# Laplace ratio
r1_lap = clip_ratio(ratio_laplace(r1_implicit_r, r1_mu_ref, r1_mu_theta))
r2_lap = clip_ratio(ratio_laplace(r2_implicit_r, r2_mu_ref, r2_mu_theta))

print("IS ratio stats:")
for name, a, b in [("Old", r1_old, r2_old), ("Normal", r1_norm, r2_norm), ("Laplace", r1_lap, r2_lap)]:
    print(f"  {name:>8}: r1 mean={np.mean(a):.4f} | r2 mean={np.mean(b):.4f}")

# ── Determine preference indicator ──
# rank=1 means response_1 preferred; chosen_idx=0 means response_1 preferred
if "rank" in ds.column_names:
    rank = ds["rank"]  # 1 if r1 preferred
elif "chosen_idx" in ds.column_names:
    rank = [1 if ci == 0 else 0 for ci in ds["chosen_idx"]]
else:
    raise KeyError(f"Need 'rank' or 'chosen_idx' column, got: {ds.column_names}")

# ── Build detail DataFrame (2 rows per pair) ──
n = len(ds)
rows = []
for i in range(n):
    is_r1_preferred = rank[i] == 1
    for ridx, tag, logp_ref, logp_dpo, logp_tgt, imp_r, kl_rd, kl_tr, kl_td, mu_r, mu_t, ro, rn, rl in [
        (0, "r1", r1_ref[i], r1_dpo[i], r1_tgt[i], r1_implicit_r[i],
         r1_kl_ref_dpo[i], r1_kl_tgt_ref[i], r1_kl_tgt_dpo[i],
         r1_mu_ref[i], r1_mu_theta[i], r1_old[i], r1_norm[i], r1_lap[i]),
        (1, "r2", r2_ref[i], r2_dpo[i], r2_tgt[i], r2_implicit_r[i],
         r2_kl_ref_dpo[i], r2_kl_tgt_ref[i], r2_kl_tgt_dpo[i],
         r2_mu_ref[i], r2_mu_theta[i], r2_old[i], r2_norm[i], r2_lap[i]),
    ]:
        rows.append({
            "pair_id": i, "response_idx": ridx,
            "is_chosen": int((ridx == 0 and is_r1_preferred) or (ridx == 1 and not is_r1_preferred)),
            "logp_ref": logp_ref, "logp_dpo": logp_dpo, "logp_target": logp_tgt,
            "implicit_reward": imp_r,
            "kl_ref_dpo": kl_rd, "kl_theta_ref": kl_tr, "kl_theta_dpo": kl_td,
            "mu_ref": mu_r, "mu_theta": mu_t,
            "ratio_old": ro, "ratio_normal": rn, "ratio_laplace": rl,
        })

df = pd.DataFrame(rows)
df.to_csv(CSV_PATH, index=False)
print(f"\nSaved {df.shape} to {CSV_PATH}")
display(df.describe())
display(df.head(4))

In [ ]:
# ═══════════════════════════════════════════
# IS preference estimation (with auto-tuned SCALE)
# ═══════════════════════════════════════════

# Estimate SCALE from empirical reward std
all_rewards = np.array(r1_implicit_r + r2_implicit_r)
est_scale = np.std(all_rewards)
print(f"Implicit reward stats: mean={np.mean(all_rewards):.4f}, std={est_scale:.4f}")
print(f"  r1: mean={np.mean(r1_implicit_r):.4f}, std={np.std(r1_implicit_r):.4f}")
print(f"  r2: mean={np.mean(r2_implicit_r):.4f}, std={np.std(r2_implicit_r):.4f}")
print(f"  Original SCALE={SCALE}, estimated SCALE={est_scale:.4f}")

# Recompute Normal and Laplace ratios with estimated scale
def ratio_normal_s(reward, mu_ref, mu_theta, scale):
    return [np.exp((-1/(2*scale**2)) * (-(r-mr)**2 + (r-mt)**2))
            for r, mr, mt in zip(reward, mu_ref, mu_theta)]

def ratio_laplace_s(reward, mu_ref, mu_theta, scale):
    return [np.exp((-1/(2*scale)) * (-abs(r-mt) + abs(r-mr)))
            for r, mr, mt in zip(reward, mu_ref, mu_theta)]

# Try multiple scales
for s in [SCALE, est_scale, est_scale * 2, est_scale * 0.5]:
    r1_n = clip_ratio(ratio_normal_s(r1_implicit_r, r1_mu_ref, r1_mu_theta, s))
    r2_n = clip_ratio(ratio_normal_s(r2_implicit_r, r2_mu_ref, r2_mu_theta, s))
    r1_l = clip_ratio(ratio_laplace_s(r1_implicit_r, r1_mu_ref, r1_mu_theta, s))
    r2_l = clip_ratio(ratio_laplace_s(r2_implicit_r, r2_mu_ref, r2_mu_theta, s))

    is_n, is_l = [], []
    for i, z in enumerate(rank):
        z = float(z)
        is_n.append(0.5 * (r1_n[i]*z + r2_n[i]*(1-z)))
        is_l.append(0.5 * (r1_l[i]*z + r2_l[i]*(1-z)))
    print(f"\n  SCALE={s:.4f}:")
    print(f"    Normal:  mean={np.mean(is_n):.4f}  std={np.std(is_n):.4f}")
    print(f"    Laplace: mean={np.mean(is_l):.4f}  std={np.std(is_l):.4f}")

# Final IS preference with best scale = estimated
print(f"\n{'='*50}")
print(f"Final IS preference (SCALE={est_scale:.4f}):")
r1_norm_est = clip_ratio(ratio_normal_s(r1_implicit_r, r1_mu_ref, r1_mu_theta, est_scale))
r2_norm_est = clip_ratio(ratio_normal_s(r2_implicit_r, r2_mu_ref, r2_mu_theta, est_scale))
r1_lap_est  = clip_ratio(ratio_laplace_s(r1_implicit_r, r1_mu_ref, r1_mu_theta, est_scale))
r2_lap_est  = clip_ratio(ratio_laplace_s(r2_implicit_r, r2_mu_ref, r2_mu_theta, est_scale))

is_old, is_norm, is_lap = [], [], []
for i, z in enumerate(rank):
    z = float(z)
    is_old.append( 0.5 * (r1_old[i]*z  + r2_old[i]*(1-z)))
    is_norm.append(0.5 * (r1_norm_est[i]*z + r2_norm_est[i]*(1-z)))
    is_lap.append( 0.5 * (r1_lap_est[i]*z  + r2_lap_est[i]*(1-z)))

print(f"  Old:     mean={np.mean(is_old):.4f}  std={np.std(is_old):.4f}")
print(f"  Normal:  mean={np.mean(is_norm):.4f}  std={np.std(is_norm):.4f}")
print(f"  Laplace: mean={np.mean(is_lap):.4f}  std={np.std(is_lap):.4f}")

In [ ]:
# ═══════════════════════════════════════════
# MC Win Rate: P(π_target > π_ref) via pairwise BT
#   = (1/N²) Σ_i Σ_j sigmoid(target_reward_i - ref_reward_j)
# ═══════════════════════════════════════════
from datasets import load_dataset
import numpy as np
import torch

mc_ds = load_dataset("august66/hh_helpfulness_mc_rewards", split="train", cache_dir=DATA_CACHE_DIR)
print(f"Loaded MC rewards: {len(mc_ds)} prompts, columns: {mc_ds.column_names}")

NUM_SAMPLES = 8  # matches generation config

# Extract reward arrays: shape (n_prompts, NUM_SAMPLES)
ref_rewards = np.array([[row[f"ref_reward_{j}"] for j in range(NUM_SAMPLES)] for row in mc_ds])
tgt_rewards = np.array([[row[f"target_reward_{j}"] for j in range(NUM_SAMPLES)] for row in mc_ds])

# Per-prompt MC win rate: average sigmoid over all N×N pairs
win_rates = []
for i in range(len(mc_ds)):
    # Outer difference: tgt_rewards[i, :, None] - ref_rewards[i, None, :] -> (N, N)
    diffs = tgt_rewards[i][:, None] - ref_rewards[i][None, :]
    probs = torch.sigmoid(torch.tensor(diffs)).numpy()
    win_rates.append(probs.mean())

win_rates = np.array(win_rates)

print(f"\nMC Win Rate  P(π_target > π_ref):")
print(f"  mean = {win_rates.mean():.6f}")
print(f"  std  = {win_rates.std():.6f}")
print(f"  min  = {win_rates.min():.6f}")
print(f"  max  = {win_rates.max():.6f}")
print(f"  median = {np.median(win_rates):.6f}")
print(f"\n  (>{0.5:.1f} means target is generally preferred over ref)")

In [10]:
# ═══════════════════════════════════════════
# Compute implicit rewards for the generation dataset
#   r(x,y) = β * (log π_dpo(y|x) - log π_ref(y|x))
# ═══════════════════════════════════════════

def prompt_key(prompt_msgs):
    """Extract a hashable key from a prompt (last user message)."""
    if isinstance(prompt_msgs, list):
        for msg in reversed(prompt_msgs):
            if isinstance(msg, dict) and msg.get("role") == "user":
                return msg["content"].strip()
    return str(prompt_msgs).strip()


    
GEN_DATASET = "august66/hh_helpfulness_qwen2.5_1.5b_generation"

gen_ds = load_dataset(GEN_DATASET, split="train", cache_dir=DATA_CACHE_DIR)
print(f"Loaded generation dataset: {len(gen_ds)} rows | Columns: {gen_ds.column_names}")

# Prepare texts (reuse helpers from earlier cells)
gen_resp1 = [get_response_text(r) for r in gen_ds["response_1"]]
gen_resp2 = [get_response_text(r) for r in gen_ds["response_2"]]
gen_prompts = [format_prompt(p) for p in tqdm(gen_ds["prompt"], desc="Gen prompts")]
gen_full1 = [format_full(p, r) for p, r in tqdm(zip(gen_ds["prompt"], gen_resp1), total=len(gen_ds), desc="Gen full r1")]
gen_full2 = [format_full(p, r) for p, r in tqdm(zip(gen_ds["prompt"], gen_resp2), total=len(gen_ds), desc="Gen full r2")]

# Log-probs under ref and DPO models
gen_r1_ref = get_sequence_logps(ref_model, gen_prompts, gen_full1, desc="Gen Ref r1")
gen_r1_dpo = get_sequence_logps(dpo_model, gen_prompts, gen_full1, desc="Gen DPO r1")
gen_r2_ref = get_sequence_logps(ref_model, gen_prompts, gen_full2, desc="Gen Ref r2")
gen_r2_dpo = get_sequence_logps(dpo_model, gen_prompts, gen_full2, desc="Gen DPO r2")

# Implicit rewards
gen_r1_implicit = [DPO_BETA * (d - r) for d, r in zip(gen_r1_dpo, gen_r1_ref)]
gen_r2_implicit = [DPO_BETA * (d - r) for d, r in zip(gen_r2_dpo, gen_r2_ref)]

print(f"\nImplicit reward stats (generation dataset):")
print(f"  r1: mean={np.mean(gen_r1_implicit):.4f}, std={np.std(gen_r1_implicit):.4f}, "
      f"min={np.min(gen_r1_implicit):.4f}, max={np.max(gen_r1_implicit):.4f}")
print(f"  r2: mean={np.mean(gen_r2_implicit):.4f}, std={np.std(gen_r2_implicit):.4f}, "
      f"min={np.min(gen_r2_implicit):.4f}, max={np.max(gen_r2_implicit):.4f}")

# Preference accuracy: does higher implicit reward match the rank label?
gen_rank = gen_ds["rank"]
correct = sum(1 for i in range(len(gen_ds))
              if (gen_r1_implicit[i] > gen_r2_implicit[i]) == (gen_rank[i] == 1))
print(f"\n  Preference accuracy (implicit reward vs rank): {correct}/{len(gen_ds)} = {correct/len(gen_ds):.4f}")

# Save to CSV
gen_df = pd.DataFrame({
    "prompt_key": [prompt_key(p) for p in gen_ds["prompt"]],
    "rank": gen_rank,
    "r1_logp_ref": gen_r1_ref, "r1_logp_dpo": gen_r1_dpo, "r1_implicit_reward": gen_r1_implicit,
    "r2_logp_ref": gen_r2_ref, "r2_logp_dpo": gen_r2_dpo, "r2_implicit_reward": gen_r2_implicit,
    "reward_diff": [r1 - r2 for r1, r2 in zip(gen_r1_implicit, gen_r2_implicit)],
})
gen_csv = "generation_implicit_rewards.csv"
gen_df.to_csv(gen_csv, index=False)
print(f"\nSaved {gen_df.shape} to {gen_csv}")
display(gen_df.describe())
display(gen_df.head(6))

Loaded generation dataset: 46070 rows | Columns: ['prompt', 'response_1', 'response_2', 'reward_1', 'reward_2', 'rank']


Gen prompts:   0%|          | 0/46070 [00:00<?, ?it/s]

Gen full r1:   0%|          | 0/46070 [00:00<?, ?it/s]

Gen full r2:   0%|          | 0/46070 [00:00<?, ?it/s]

Gen Ref r1:   0%|          | 0/11518 [00:00<?, ?it/s]

...

Gen Ref r2:   0%|          | 0/11518 [00:00<?, ?it/s]

Gen DPO r2:   0%|          | 0/11518 [00:00<?, ?it/s]


Implicit reward stats (generation dataset):
  r1: mean=-4.3912, std=3.8182, min=-36.8000, max=1.1000
  r2: mean=-4.5237, std=3.7883, min=-37.2000, max=4.4000

  Preference accuracy (implicit reward vs rank): 28415/46070 = 0.6168

Saved (46070, 9) to generation_implicit_rewards.csv


,rank,r1_logp_ref,r1_logp_dpo,r1_implicit_reward,r2_logp_ref,r2_logp_dpo,r2_implicit_reward,reward_diff
count,46070.000000,46070.000000,46070.000000,46070.000000,46070.000000,46070.000000,46070.000000,46070.000000
mean,0.567984,-152.974494,-196.886732,-4.391224,-140.666286,-185.903496,-4.523721,0.132497
std,0.495362,104.802233,126.098137,3.818198,96.971174,119.180290,3.788314,2.311152
min,0.000000,-2144.000000,-2192.000000,-36.800000,-1256.000000,-1272.000000,-37.200000,-24.840430
25%,0.000000,-201.000000,-264.000000,-6.500000,-186.000000,-248.000000,-6.600000,-0.500000
50%,1.000000,-129.000000,-172.000000,-3.600000,-119.500000,-161.000000,-3.850000,0.150000
75%,1.000000,-80.000000,-105.000000,-1.250000,-73.000000,-103.000000,-1.350000,0.850000
max,1.000000,-0.158203,-0.484375,1.100000,-0.158203,-0.484375,4.400000,22.150000


,prompt_key,rank,r1_logp_ref,r1_logp_dpo,r1_implicit_reward,r2_logp_ref,r2_logp_dpo,r2_implicit_reward,reward_diff
0,"Okay. What else is needed to play, and what ar...",0,-137.0,-149.0,-1.20,-188.0,-227.0,-3.9,2.70
1,any other ideas? they are fidgeting,0,-90.5,-108.0,-1.75,-165.0,-182.0,-1.7,-0.05
2,Thanks. And what are some typical forms required?,0,-71.0,-81.5,-1.05,-141.0,-138.0,0.3,-1.35
3,How can I store food if I don't have a pantry?,1,-96.5,-139.0,-4.25,-57.0,-111.0,-5.4,1.15
4,That sounds cool,0,-178.0,-246.0,-6.80,-173.0,-235.0,-6.2,-0.60
5,Then why do they sell it?,1,-224.0,-336.0,-11.20,-172.0,-282.0,-11.0,-0.20


In [1]:
# ═══════════════════════════════════════════
# Download & cache august66/hh_qwen_1.5b_sft_dpo_generation
# ═══════════════════════════════════════════
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "august66/hh_qwen_1.5b_sft_dpo_generation"
MODEL_CACHE_DIR = "/root/autodl-tmp/model_cache"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, trust_remote_code=True, cache_dir=MODEL_CACHE_DIR,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True, cache_dir=MODEL_CACHE_DIR,
)
print(f"Model & tokenizer saved to {MODEL_CACHE_DIR}")
del model, tokenizer
torch.cuda.empty_cache()

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Model & tokenizer saved to /root/autodl-tmp/model_cache
